# Logistic Regression as a Neural Network

> Based on Stanford CS230 — C1M2, Andrew Ng / DeepLearning.AI

Logistic regression is the **simplest neural network** — a single neuron. Understanding it deeply (forward pass, loss, gradients) is the foundation for everything in deep learning.

## Learning Objectives

1. Formulate binary classification as a logistic regression problem
2. Derive the logistic regression loss function from maximum likelihood
3. Implement gradient descent for logistic regression from scratch
4. Understand how vectorisation replaces explicit loops

## Problem Setup — Binary Classification

Given an input $x \in \mathbb{R}^{n_x}$, predict a binary label $y \in \{0, 1\}$.

**Notation:**
- $m$ training examples: $(x^{(1)}, y^{(1)}), \ldots, (x^{(m)}, y^{(m)})$
- Input matrix: $X \in \mathbb{R}^{n_x \times m}$ — each column is one example
- Label vector: $Y \in \mathbb{R}^{1 \times m}$


## The Model

We want $\hat{y} = P(y=1 \mid x)$. We need an output in $[0,1]$, so we use the **sigmoid**:

$$z = w^T x + b, \qquad \hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Parameters: $w \in \mathbb{R}^{n_x}$, $b \in \mathbb{R}$.

## Loss Function

We cannot use squared error $(\hat{y} - y)^2$ — it is non-convex for logistic regression. Instead, derived from MLE (Bernoulli likelihood):

$$\mathcal{L}(\hat{y}, y) = -\bigl[y \log \hat{y} + (1-y)\log(1-\hat{y})\bigr]$$

- When $y=1$: we want $\hat{y}$ large → minimising $-\log \hat{y}$ pushes $\hat{y} \to 1$
- When $y=0$: we want $\hat{y}$ small → minimising $-\log(1-\hat{y})$ pushes $\hat{y} \to 0$

**Cost** (average over $m$ examples):

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)})$$

## Gradient Descent

To minimise $J$, update parameters iteratively:

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}, \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

where $\alpha$ is the **learning rate**.

**Key gradients** (derived via chain rule):

$$\frac{\partial \mathcal{L}}{\partial z} = \hat{y} - y, \qquad \frac{\partial J}{\partial w} = \frac{1}{m} X(\hat{Y} - Y)^T, \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\sum_i (\hat{y}^{(i)} - y^{(i)})$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# --- Visualise sigmoid and loss ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Logistic Regression as a Neural Network', fontsize=14, fontweight='bold')

z = np.linspace(-6, 6, 400)
sigma = 1 / (1 + np.exp(-z))

# Sigmoid
axes[0].plot(z, sigma, color=P[0], lw=2.5)
axes[0].axhline(0.5, color=P[2], lw=1, ls='--', label='σ(0) = 0.5')
axes[0].axvline(0, color='#888', lw=0.8, ls=':')
axes[0].set_xlabel('z = wᵀx + b')
axes[0].set_ylabel('σ(z)')
axes[0].set_title('Sigmoid Function')
axes[0].legend(fontsize=9)
axes[0].grid(True)

# Loss curves
yhat = np.linspace(0.001, 0.999, 300)
axes[1].plot(yhat, -np.log(yhat),     color=P[1], lw=2.5, label='y=1:  −log(ŷ)')
axes[1].plot(yhat, -np.log(1 - yhat), color=P[0], lw=2.5, label='y=0:  −log(1−ŷ)')
axes[1].set_xlabel('ŷ (predicted probability)')
axes[1].set_ylabel('Loss ℒ(ŷ, y)')
axes[1].set_title('Loss Function')
axes[1].set_ylim(0, 5)
axes[1].legend(fontsize=9)
axes[1].grid(True)

# From-scratch gradient descent on toy data
np.random.seed(42)
n, m = 2, 200
X = np.vstack([np.random.randn(2, m//2) + np.array([[2], [2]]),
               np.random.randn(2, m//2) + np.array([[-2], [-2]])]).T
X = X.T  # (2, m)
Y = np.array([1]*(m//2) + [0]*(m//2)).reshape(1, -1)

w = np.zeros((n, 1))
b = 0
alpha = 0.3
costs = []

for _ in range(500):
    Z = w.T @ X + b
    A = 1 / (1 + np.exp(-Z))
    cost = -np.mean(Y * np.log(A + 1e-8) + (1 - Y) * np.log(1 - A + 1e-8))
    dZ = A - Y
    dw = (X @ dZ.T) / m
    db = np.mean(dZ)
    w -= alpha * dw
    b -= alpha * db
    costs.append(cost)

axes[2].plot(costs, color=P[3], lw=2)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Cost J(w, b)')
axes[2].set_title(f'Gradient Descent (α={alpha})\nFinal cost: {costs[-1]:.4f}')
axes[2].grid(True)

plt.tight_layout()
plt.show()

# Decision boundary
fig, ax = plt.subplots(figsize=(6, 5))
ax.set_facecolor('#ffffff')
ax.set_title('Learned Decision Boundary', fontsize=12)
ax.scatter(X[0, Y[0]==1], X[1, Y[0]==1], color=P[0], s=25, alpha=0.7, label='y=1')
ax.scatter(X[0, Y[0]==0], X[1, Y[0]==0], color=P[1], s=25, alpha=0.7, label='y=0')

x1_vals = np.linspace(-5, 5, 200)
x2_vals = -(w[0, 0] * x1_vals + b) / (w[1, 0] + 1e-8)
ax.plot(x1_vals, x2_vals, color=P[2], lw=2.5, label='Decision boundary')
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

A_final = 1 / (1 + np.exp(-(w.T @ X + b)))
preds = (A_final > 0.5).astype(int)
acc = np.mean(preds == Y)
print(f"Training accuracy: {acc*100:.1f}%")
print(f"Learned w: {w.flatten()}, b: {b:.4f}")


## Vectorisation — Why It Matters

**Without vectorisation** (explicit loop over examples):
```python
for i in range(m):
    z[i] = np.dot(w.T, x[:, i]) + b
    a[i] = sigmoid(z[i])
    dw += x[:, i] * (a[i] - y[i])
    db += (a[i] - y[i])
dw /= m; db /= m
```

**With vectorisation** (single NumPy operation):
```python
Z = w.T @ X + b          # (1, m)
A = sigmoid(Z)            # (1, m)
dw = (X @ (A - Y).T) / m  # (n_x, 1)
db = np.mean(A - Y)       # scalar
```

The vectorised version runs **100–1000× faster** because NumPy (and PyTorch/TensorFlow) dispatch to optimised BLAS routines and can use SIMD / GPU parallelism.

> **Rule of thumb:** whenever you see a `for i in range(m)` loop in ML code, ask if it can be replaced by a matrix operation.
